### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [30]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config
import os

default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cpu',
    default_dtype="float32",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)
import numpy as np
import random
import torch
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    #print(f"Random seed set as {seed}")

os.environ['NEQUIP_NUM_TASKS'] = '4'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [31]:
config = Config.from_file('./config/example_ETN_opt_MEA.yaml', defaults=default_config)

config['root'] = 'results/MEA_Allegro_10'
config['seed'] = 123456 + 8 #+ 8
set_seed(config['seed'])
torch.manual_seed(config['seed'])
dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

search for AtomicData_options with prefix dataset
search for r_max with prefix dataset
          0_args :                                               r_max
instantiate TypeMapper
   optional_args :                             chemical_symbol_to_type
...TypeMapper_param = dict(
...   optional_args = {'type_names': None, 'chemical_symbol_to_type': {'Nb': 0, 'Mo': 1, 'Ta': 2, 'W': 3}, 'type_to_chemical_symbol': None, 'chemical_symbols': None},
...   positional_args = {})
instantiate register_fields
...register_fields_param = dict(
...   optional_args = {'node_fields': [], 'edge_fields': [], 'graph_fields': [], 'long_fields': []},
...   positional_args = {})
instantiate ASEDataset
   optional_args :                                                root
   optional_args :                                            ase_args
   optional_args :                                           file_name <-                                  dataset_file_name
   optional_args :                           

In [32]:
dataset

ASEDataset(5529)

In [33]:
config['seed']

123464

In [34]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

config['seed'] = 123456 + 8 #+ 8

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
#Nc = 10 # number of chennels for F features from ETN paper
#N_rank_spec = 4 # hidden rank of reduction for type radial tensor
#config['Nc'] = Nc
#config['N_rank_spec'] = N_rank_spec

# ETN parameters
#config['d'] = 4 # dimention of the tensor train
#config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train)

* Initialize Trainer
* Initialize Output
  ...generate file name results/MEA_Allegro_10/example/log
  ...open log file results/MEA_Allegro_10/example/log
  ...generate file name results/MEA_Allegro_10/example/metrics_epoch.csv
  ...open log file results/MEA_Allegro_10/example/metrics_epoch.csv
  ...generate file name results/MEA_Allegro_10/example/metrics_initialization.csv
  ...open log file results/MEA_Allegro_10/example/metrics_initialization.csv
  ...generate file name results/MEA_Allegro_10/example/metrics_batch_train.csv
  ...open log file results/MEA_Allegro_10/example/metrics_batch_train.csv
  ...generate file name results/MEA_Allegro_10/example/metrics_batch_val.csv
  ...open log file results/MEA_Allegro_10/example/metrics_batch_val.csv
  ...generate file name results/MEA_Allegro_10/example/best_model.pth
  ...generate file name results/MEA_Allegro_10/example/last_model.pth
  ...generate file name results/MEA_Allegro_10/example/trainer.pth
  ...generate file name results/MEA_A

...   positional_args = {'irreps_in': {'pos': 1x1oe, 'edge_index': None, 'edge_types': 1x0ee, 'node_attrs': 4x0ee, 'node_features': 4x0ee, 'edge_embedding': 8x0ee, 'edge_cutoff': 1x0ee, 'edge_attrs': 1x0ee+1x1oe+1x2ee, 'edge_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_ETN': 10x0ee+10x1oe+10x2ee, 'atomic_energy': 1x0ee}})


cpu


Replace string dataset_forces_rms to 0.8583642840385437
Initially outputs are globally scaled by: 0.8583642840385437, total_energy are globally shifted by None.
PerSpeciesScaleShift's arguments were in dataset units; rescaling:
  Original scales: [Nb: 0.858364, Mo: 0.858364, Ta: 0.858364, W: 0.858364] shifts: [Nb: -11.415697, Mo: -11.415697, Ta: -11.415697, W: -11.415697]
  New scales: [Nb: 1.000000, Mo: 1.000000, Ta: 1.000000, W: 1.000000] shifts: [Nb: -13.299362, Mo: -13.299362, Ta: -13.299362, W: -13.299362]


In [35]:
final_model.get_submodule('model.model.func.etn.cores').requires_grad_(True)

ParameterList(
    (0): Parameter containing: [torch.float32 of size 3x10x4]
    (1): Parameter containing: [torch.float32 of size 4x10x4x11]
    (2): Parameter containing: [torch.float32 of size 4x10x4x11]
    (3): Parameter containing: [torch.float32 of size 3x4x10]
)

In [36]:
trainer.device

'cpu'

In [37]:
final_model.get_submodule('model.model.func.etn.cores')[1]

Parameter containing:
tensor([[[[-6.9987e-01,  2.9879e-01, -1.1493e+00,  ..., -1.9035e-01,
            2.7405e-01, -8.2327e-01],
          [-1.1407e+00,  1.2598e+00,  6.3309e-01,  ...,  2.9976e+00,
            5.5204e-01,  2.1799e-01],
          [-1.2566e+00,  8.6083e-01,  1.3862e-01,  ...,  7.4154e-01,
            2.4280e+00,  1.3661e-01],
          [ 1.8345e-01,  1.1931e+00,  4.9321e-02,  ...,  4.7729e-01,
           -1.4764e+00,  1.9577e+00]],

         [[ 1.6494e+00,  1.0561e+00,  1.6840e+00,  ..., -4.6733e-01,
            2.0822e+00, -7.4287e-02],
          [ 1.2003e+00, -1.0893e+00,  8.2627e-01,  ...,  5.5658e-01,
           -1.3207e+00, -1.5904e-01],
          [-4.8256e-01, -2.7152e-01, -7.9485e-01,  ..., -6.4076e-01,
            3.8341e-01,  1.2064e+00],
          [-1.1419e+00,  1.3489e+00, -4.3856e-01,  ..., -1.3762e+00,
            8.8529e-02, -6.5544e-01]],

         [[-2.7744e-01, -2.7149e-01,  3.0107e-01,  ..., -1.5926e-01,
            5.5303e-01, -2.5469e-01],
          [

In [38]:
# continue

In [8]:
from nequip.data import AtomicData, AtomicDataDict


In [29]:
from nequip.data import AtomicData, AtomicDataDict

seed = -5000

# Set seed
config['seed'] = 123456 + seed #+ 8

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)


# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train)

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[20])

# forward pass
data_new = final_model(data0)



* Initialize Trainer
* Initialize Output
  ...generate file name results/MEA_Allegro_10/example/log
  ...open log file results/MEA_Allegro_10/example/log
  ...generate file name results/MEA_Allegro_10/example/metrics_epoch.csv
  ...open log file results/MEA_Allegro_10/example/metrics_epoch.csv
  ...generate file name results/MEA_Allegro_10/example/metrics_initialization.csv
  ...open log file results/MEA_Allegro_10/example/metrics_initialization.csv
  ...generate file name results/MEA_Allegro_10/example/metrics_batch_train.csv
  ...open log file results/MEA_Allegro_10/example/metrics_batch_train.csv
  ...generate file name results/MEA_Allegro_10/example/metrics_batch_val.csv
  ...open log file results/MEA_Allegro_10/example/metrics_batch_val.csv
  ...generate file name results/MEA_Allegro_10/example/best_model.pth
  ...generate file name results/MEA_Allegro_10/example/last_model.pth
  ...generate file name results/MEA_Allegro_10/example/trainer.pth
  ...generate file name results/MEA_A

cpu


Replace string dataset_forces_rms to 0.8583642840385437
Replace string dataset_per_atom_total_energy_mean to -11.41569709777832
Atomic outputs are scaled by: [Nb, Mo, Ta, W: 0.858364], shifted by [Nb, Mo, Ta, W: -11.415697].
instantiate PerSpeciesScaleShift
        all_args :                                       default_dtype
        all_args :                                           num_types
        all_args :                                          type_names
   optional_args :                          arguments_in_dataset_units
   optional_args :                                               field
   optional_args :                                              shifts
   optional_args :                                           out_field
   optional_args :                                              scales
...PerSpeciesScaleShift_param = dict(
...   optional_args = {'out_field': 'atomic_energy', 'scales_trainable': False, 'shifts_trainable': False, 'default_dtype': 'float32', '

NotImplementedError: Could not run 'aten::empty.memory_format' with arguments from the 'SparseMPS' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::empty.memory_format' is only available for these backends: [CPU, MPS, Meta, QuantizedCPU, QuantizedMeta, MkldnnCPU, SparseCPU, SparseMeta, SparseCsrCPU, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMeta, AutogradMTIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradNestedTensor, Tracer, AutocastCPU, AutocastCUDA, FuncTorchBatched, FuncTorchVmapMode, Batc

In [26]:
data_new[_keys.NODE_FEATURES_ETN].device

device(type='cpu')

In [25]:
from allegro import _keys

data_new[_keys.NODE_FEATURES_ETN][7][:, 0]

tensor([ 4.4474e-04, -1.9308e-05,  6.7328e-06,  6.2771e-06,  2.6270e-04,
        -9.9475e-05,  7.0377e-05,  1.1085e-04, -8.8745e-05],
       grad_fn=<SelectBackward0>)

In [11]:
data_new['total_energy']

tensor([[-718.1641]], grad_fn=<ScatterAddBackward0>)

In [27]:
trainer.train()

NotImplementedError: Could not run 'aten::empty.memory_format' with arguments from the 'SparseMPS' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::empty.memory_format' is only available for these backends: [CPU, MPS, Meta, QuantizedCPU, QuantizedMeta, MkldnnCPU, SparseCPU, SparseMeta, SparseCsrCPU, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMeta, AutogradMTIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradNestedTensor, Tracer, AutocastCPU, AutocastCUDA, FuncTorchBatched, FuncTorchVmapMode, Batc

In [ ]:
 1   100        0.757        0.639        0.118        0.371        0.686         5.29         0.23
    
1   100         1.23        0.933        0.293         0.41        0.829         4.24        0.315
1   100         2.14         1.73        0.413        0.608         1.13         6.13        0.412

In [15]:
config

{'_jit_bailout_depth': 2, '_jit_fusion_strategy': [('DYNAMIC', 3)], '_jit_fuser': 'fuser1', 'root': 'results/MEA_Allegro_1', 'tensorboard': False, 'wandb': False, 'model_builders': ['allegro.model.ETN_opt', 'PerSpeciesRescale', 'ParaStressForceOutput', 'RescaleEnergyEtc'], 'dataset_statistics_stride': 1, 'device': 'cuda:0', 'default_dtype': 'float32', 'model_dtype': 'float32', 'allow_tf32': True, 'verbose': 'debug', 'model_debug_mode': False, 'equivariance_test': False, 'grad_anomaly_mode': False, 'gpu_oom_offload': False, 'append': True, 'warn_unused': False, 'run_name': 'example', 'seed': 123458, 'dataset_seed': 123456, 'r_max': 5.0, 'Nc': 10, 'd': 4, 'N_rank_spec': 4, 'N_rank_ett': [4, 4, 4], 'avg_num_neighbors': 26.18090057373047, 'BesselBasis_trainable': True, 'PolynomialCutoff_p': 5, 'l_max': 2, 'parity': 'o3_full', 'num_layers': 2, 'env_embed_multiplicity': 64, 'embed_initial_edge': True, 'two_body_latent_mlp_latent_dimensions': [128, 256, 512, 1024], 'two_body_latent_mlp_nonlin

In [10]:
def batch_step(data, validation=False):
    # no need to have gradients from old steps taking up memory
    #self.optim.zero_grad(set_to_none=True)

    #if validation:
    #    self.model.eval()
    #else:
    #    self.model.train()

    # Do any target rescaling
    data = AtomicData.to_AtomicDataDict(data)

    # this will normalize the targets
    # in both validation and train we want targets normalized _for the loss_
    data_for_loss = trainer.model.unscale(data, force_process=True)

    # Run model
    # We make a shallow copy of the input dict in case the model modifies it
    out = trainer.model(data_for_loss)
    #print(out)
    return out

In [12]:
from nequip.train._key import ABBREV, LOSS_KEY, TRAIN, VALIDATION

def epoch_step(trainer):

    dataloaders = {TRAIN: trainer.dl_train, VALIDATION: trainer.dl_val}
    categories = [TRAIN, VALIDATION] if trainer.iepoch >= 0 else [VALIDATION]
    dataloaders = [
        dataloaders[c] for c in categories
    ]  # get the right dataloaders for the catagories we actually run
    if TRAIN in categories:
        # We have to step the sampler so it knows what epoch it is
        trainer.dl_train_sampler.step_epoch(trainer.iepoch)

    #self.metrics_dict = {}
    #self.loss_dict = {}

    for category, dataset in zip(categories, dataloaders):
        
        for trainer.ibatch, batch in enumerate(dataset):
            print(trainer.ibatch, batch)
            out = batch_step(
                data=batch,
                validation=(category == VALIDATION),
            )
            
    return out, batch

In [13]:
len(dataset[:20])

20

In [14]:
dataset[:20]

ASEDataset(20)

In [15]:
trainer.n_train

5000

In [16]:
trainer.dl_val.batch_size

5

In [19]:
trainer.n_train = 2
trainer.n_val = 3

trainer.train_idcs = torch.tensor([201, 1201], dtype = torch.long)
trainer.val_idcs = torch.tensor([301, 1401, 5], dtype = torch.long)

trainer.set_dataset(dataset, None)

In [20]:
out, batch = epoch_step(trainer)

0 Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])


In [21]:
out['pos'].shape

torch.Size([34, 3])

In [22]:
out['batch']

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 2, 2])

In [23]:
out['ptr']

tensor([ 0, 16, 32, 34])

In [24]:
print(AtomicData.to_AtomicDataDict(dataset[301])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[1401])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[5])['pos'].shape)

torch.Size([16, 3])
torch.Size([16, 3])
torch.Size([2, 3])


In [25]:
out.keys()

dict_keys(['edge_index', 'pos', 'batch', 'ptr', 'cell', 'edge_cell_shift', 'atom_types', 'edge_vectors', 'edge_types', 'node_attrs', 'node_features', 'edge_lengths', 'edge_embedding', 'edge_cutoff', 'edge_attrs', 'edge_features_F', 'node_features_F', 'node_features_ETN', 'atomic_energy', 'total_energy', 'forces', 'stress', 'virial', 'atom_virial'])

In [28]:
trainer.batch_metrics = trainer.metrics(pred=out, ref=batch)

ValueError: Data shape of batch, torch.Size([3, 34]), does not match the input data dimension of this RunningStats, torch.Size([192])

In [29]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [30]:
for key in out:
    print(key, out[key].shape)

edge_index torch.Size([2, 884])
pos torch.Size([34, 3])
batch torch.Size([34])
ptr torch.Size([4])
cell torch.Size([3, 3, 3])
edge_cell_shift torch.Size([884, 3])
atom_types torch.Size([34, 1])
edge_vectors torch.Size([884, 3])
edge_types torch.Size([884, 1])
node_attrs torch.Size([34, 4])
node_features torch.Size([34, 4])
edge_lengths torch.Size([884])
edge_embedding torch.Size([884, 8])
edge_cutoff torch.Size([884, 1])
edge_attrs torch.Size([884, 9])
edge_features_F torch.Size([884, 9, 10])
node_features_F torch.Size([34, 9, 10])
node_features_ETN torch.Size([34, 9, 10])
atomic_energy torch.Size([34, 34])
total_energy torch.Size([3, 34])
forces torch.Size([34, 3])
stress torch.Size([3, 3, 3])
virial torch.Size([3, 3, 3])
atom_virial torch.Size([34, 3, 3])


In [31]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [32]:
out['total_energy'].shape

torch.Size([3, 34])

In [40]:

( data_new[_keys.NODE_FEATURES_ETN] * data_new[_keys.NODE_FEATURES_ETN] ).sum(dim = (-2, -1)).unsqueeze(-1)

tensor([[5.7274e-15],
        [5.7274e-15]], grad_fn=<UnsqueezeBackward0>)

In [37]:
data_new["node_features_F"].shape

torch.Size([2, 9, 10])

In [32]:
from allegro.nn._etn_opt import ETN_Module_opt

torch.manual_seed(184)

ETN = ETN_Module_opt(d = config['d'],
                         N_rank_ett = config['N_rank_ett'], 
                         irreps_in = final_model.irreps_out,
                         out_field = AtomicDataDict.PER_ATOM_ENERGY_KEY)

cpu


/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(


In [31]:
data_new_new = ETN(data_new)

data_new_new['atomic_energy'][0]

tensor([1.9666e-10], grad_fn=<SelectBackward0>)

In [33]:
data_new_new = ETN(data_new)
data_new_new['atomic_energy'][0]

tensor([-1.3933e-09], grad_fn=<SelectBackward0>)